# 01 - Transport of a pollutant: advection, diffusion, removal

This notebook runs the forward model of **pravah**: how a pollutant released from a few sources is carried by the
wind, spreads out, and slowly disappears. Everything here is an **idealised synthetic experiment**. The numbers are
demonstration values, not estimates for any real place (including Belauri, Nepal).

## The equation

The concentration $c(x,y,t)$ (micrograms per cubic metre, $\mu\mathrm{g\,m^{-3}}$) obeys

$$
\frac{\partial c}{\partial t} + \nabla\!\cdot(\mathbf u\,c)
= \nabla\!\cdot(K\nabla c) + \sum_j q_j\,s_j(t)\,\frac{f_j(x,y)}{H} - \lambda c .
$$

| symbol | meaning | unit |
|---|---|---|
| $x,y$ | position (east, north) | m |
| $t$ | time | s |
| $\mathbf u=(u,v)$ | wind velocity, uniform in space, piecewise constant in time | m/s |
| $K$ | eddy diffusivity (turbulent spreading) | m$^2$/s |
| $\lambda$ | first-order removal rate (deposition, chemistry); lifetime $1/\lambda$ | 1/s |
| $H$ | effective mixing depth, fixed | m |
| $q_j$ | emission rate of source $j$ | $\mu$g/s |
| $f_j$ | Gaussian shape of source $j$, normalised so that $\sum f_j\,\Delta A = 1$ | 1/m$^2$ |
| $s_j(t)$ | dimensionless on/off profile of source $j$ (default 1) | - |

In words: the concentration at a point changes because (1) wind carries pollutant in or out, (2) turbulence
spreads it from high to low values, (3) the sources add pollutant (the source rate is divided by $H$ because we model
a single well-mixed layer, so $\mu$g/s becomes $\mu$g m$^{-3}$ s$^{-1}$), and (4) removal destroys a fraction
$\lambda$ of it per second.

## Sign convention and boundaries

Let $\mathbf n$ be the **outward** normal of a domain edge. An edge (face) is **inflow** where $\mathbf u\cdot\mathbf n<0$
(wind enters the domain), **outflow** where $\mathbf u\cdot\mathbf n>0$, and tangential where $\mathbf u\cdot\mathbf n=0$.
With the default *open* boundary: on inflow faces the incoming air has a prescribed concentration $c_b$ (here 0,
clean air); on outflow faces pollutant simply leaves with the wind and there is no diffusive gradient; tangential
faces carry no diffusive flux. Because the wind direction can change, which faces are inflow and which are
outflow can change too (demo d).

## What is idealised

* Uniform wind, constant $K$, constant $\lambda$: no terrain, no stability changes, no chemistry beyond one decay rate.
* One well-mixed layer of fixed depth $H$ (a 2-D model).
* Sources are smooth Gaussian blobs, not point sources or real inventories.
* Flat clean-air background and a clean initial state.
* A finite-volume solver (FiPy) with backward Euler time stepping: stable, but it smooths sharp features.

The mass accounting below is exact for the discrete scheme, so it is a good check that nothing is lost or invented.

## Parameters
Change values in the next cell and re-run everything below it. Defaults come from `configs/reference.json`.

In [ ]:
# ---- PARAMETER CELL: edit these values ------------------------------------------------------------------------------
from pathlib import Path
import dataclasses

from pravah.config import (Domain, GaussianSource, Scenario, SolverOptions, Times, Transport, WindSegment,
                           load_config, scenario_from_dict)

# find the repository root relative to this notebook (works from the repo root or from notebooks/)
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "configs" / "reference.json").exists())
CONFIGS = ROOT / "configs"
_ref = load_config(CONFIGS / "reference.json")          # defaults for everything below
_sc = _ref["scenario"]

LX, LY, NX, NY = (_sc["domain"][k] for k in ("lx", "ly", "nx", "ny"))   # domain size (m) and number of cells
WIND_U, WIND_V = _sc["transport"]["wind"][0]["u"], _sc["transport"]["wind"][0]["v"]   # wind (m/s)
K_DIFF = _sc["transport"]["K"]            # eddy diffusivity (m^2/s)
LAMBDA = _sc["transport"]["lam"]          # removal rate (1/s); lifetime = 1/LAMBDA
H_MIX = _sc["transport"]["H"]             # mixing depth (m)
SOURCES = [(s["name"], s["x"], s["y"], s["sigma"]) for s in _sc["sources"]]   # (name, x m, y m, sigma m)
Q = list(_ref["q"])                       # emission rates (microgram/s), one per source
DURATION = _sc["times"]["duration"]       # simulated time (s)
# ---------------------------------------------------------------------------------------------------------------------

DT, OUT_INT = _sc["times"]["dt"], _sc["times"]["output_interval"]
SOLVER = SolverOptions(**_ref["solver"])
scenario = Scenario(
    domain=Domain(LX, LY, NX, NY),
    sources=tuple(GaussianSource(n, x, y, s) for n, x, y, s in SOURCES),
    transport=Transport(wind=(WindSegment(0.0, WIND_U, WIND_V),), K=K_DIFF, lam=LAMBDA, H=H_MIX),
    times=Times(duration=DURATION, dt=DT, output_interval=OUT_INT),
)
print(f"{NX}x{NY} cells of {scenario.domain.dx:.0f} m, wind ({WIND_U}, {WIND_V}) m/s, K={K_DIFF} m^2/s, "
      f"lifetime 1/lambda = {1/LAMBDA/3600:.1f} h, H={H_MIX} m, q={Q} microgram/s, {DURATION:.0f} s")

## Run the simulation and look at the sources

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import HTML, display

from pravah.forward import simulate
from pravah.plotting import (animate_field, plot_field_snapshots, plot_mass_budget, plot_sources, plot_traces)
from pravah.observations import SensorLayout, sample

def show(fig):
    display(fig)
    plt.close(fig)

result = simulate(scenario, Q, SOLVER)
conc = result.concentration
print(f"status: {result.status}; solved in {result.runtime_s:.1f} s; {result.diagnostics['n_steps']} time steps")
show(plot_sources(scenario.sources, scenario.domain))

## Concentration snapshots
The wind blows towards +x and slightly +y, so each blob is stretched into a plume that drifts off to the upper right.

In [ ]:
show(plot_field_snapshots(conc, times=[900.0, 3600.0, 7200.0, DURATION]))

## Animation
At most 49 frames are drawn so the notebook stays light. If animation output is not available the cell prints a message instead.

In [ ]:
MAX_FRAMES = 49
try:
    step = int(np.ceil(conc.sizes["time"] / MAX_FRAMES))
    anim = animate_field(conc.isel(time=slice(0, None, step)))
    html = anim.to_jshtml()
    plt.close(anim._fig)
    display(HTML(html))
except Exception as exc:   # animation is a convenience only
    print("Animation unavailable:", type(exc).__name__, exc)

## Point traces
Concentration over time at three locations (bilinear interpolation of the grid, as a sensor would see it).

In [ ]:
spots = SensorLayout(ids=("A", "B", "C"), x=(4000.0, 5500.0, 8000.0), y=(3000.0, 5000.0, 4500.0))
show(plot_traces(sample(conc, spots, conc["time"].values[1:])))

## Mass budget

Total mass in the layer is $M=H\sum c\,\Delta A$. For each output interval the model records
$\Delta M = \text{emission} - \text{removal} + \text{inflow} - \text{outflow}$. The **residual** (difference of the two
sides) should be at round-off level. A conservative scheme can still produce small negative concentrations; they are
**never clipped**, only counted.

In [ ]:
b = result.budget
show(plot_mass_budget(b))
d = result.diagnostics
print(f"total emitted : {b['emission'].sum().item():.4e} microgram")
print(f"total removed : {b['removal'].sum().item():.4e} microgram")
print(f"total outflow : {b['outflow'].sum().item():.4e} microgram  (inflow {b['inflow'].sum().item():.2e})")
print(f"final mass    : {(b['mass_change'].sum()).item():.4e} microgram")
print(f"max |residual| per interval : {d['max_abs_budget_residual']:.3e} microgram "
      f"(relative to the largest term: {d['max_rel_budget_residual']:.1e})")
print(f"negative values: {d['n_negative']} of {conc.size} samples; minimum concentration {d['min_concentration']:.3e}")

## Demos: change one thing and compare with the reference

The helper below runs a modified scenario and plots, at the end of the run, the reference field, the modified field,
their difference, and the total mass in the layer over time.

In [ ]:
def total_mass(c):
    return H_MIX * c.sum(("x", "y")) * scenario.domain.dx * scenario.domain.dy

def compare(title, alt_scenario, alt_q):
    alt = simulate(alt_scenario, alt_q, SOLVER)
    a, b_ = conc.isel(time=-1).values, alt.concentration.isel(time=-1).values
    vmax = max(a.max(), b_.max())
    ext = (0, LX, 0, LY)
    fig, ax = plt.subplots(1, 4, figsize=(17, 3.6), constrained_layout=True)
    for k, (arr, name) in enumerate([(a, "reference"), (b_, title)]):
        im = ax[k].imshow(arr, origin="lower", extent=ext, vmin=0, vmax=vmax)
        ax[k].set(title=name, xlabel="x (m)", ylabel="y (m)")
    fig.colorbar(im, ax=ax[:2], label="concentration (microgram/m$^3$)")
    m = np.abs(b_ - a).max() or 1.0
    im2 = ax[2].imshow(b_ - a, origin="lower", extent=ext, cmap="RdBu_r", vmin=-m, vmax=m)
    ax[2].set(title="modified - reference", xlabel="x (m)")
    fig.colorbar(im2, ax=ax[2])
    ax[3].plot(conc["time"], total_mass(conc), label="reference")
    ax[3].plot(alt.concentration["time"], total_mass(alt.concentration), label=title)
    ax[3].set(title="total mass in layer", xlabel="t (s)", ylabel="microgram")
    ax[3].legend()
    show(fig)
    print(f"peak: reference {a.max():.3g}, modified {b_.max():.3g} microgram/m^3; "
          f"final mass: reference {total_mass(conc).values[-1]:.3g}, modified {total_mass(alt.concentration).values[-1]:.3g} microgram")
    return alt

def with_transport(**changes):
    return dataclasses.replace(scenario, transport=dataclasses.replace(scenario.transport, **changes))

### (a) Change the wind
Turn the wind towards the north-east (u=0.4, v=0.9 m/s): plumes now move mostly northwards.

In [ ]:
_ = compare("wind (0.4, 0.9)", with_transport(wind=(WindSegment(0.0, 0.4, 0.9),)), Q)

### (b) Change source strength and position
Double S1, and move S3 from (6500, 3500) to (6500, 7000).

In [ ]:
srcs = list(scenario.sources)
srcs[2] = dataclasses.replace(srcs[2], y=7000.0)
q_b = [2 * Q[0], Q[1], Q[2]]
_ = compare("S1 x2, S3 moved", dataclasses.replace(scenario, sources=tuple(srcs)), q_b)

### (c) Change the removal rate
Five times faster removal (lifetime about 1.2 h instead of 6 h): plumes are shorter and less mass accumulates.

In [ ]:
_ = compare("5x removal", with_transport(lam=5 * LAMBDA), Q)

### (d) A wind that changes in time

`configs/wind_change.json` extends the reference config (`"extends": "reference.json"`) and replaces only the wind by
piecewise-constant segments: (1.0, 0.3) m/s until 5400 s, then (-0.6, 0.8) m/s until 10200 s, then (0.9, -0.4) m/s.
In the budget plot, the dashed lines mark the switches. After a switch, different faces become inflow or outflow, so
the boundary terms jump. The incoming air is clean (=0$), so the **inflow** term is never positive: it is
slightly negative when pollutant diffuses out through an inflow face. It becomes strongly negative right after a switch,
when a plume that had been sitting at an outflow edge suddenly finds itself on an inflow edge. The **outflow** term
drops at the same moments (the plumes are pushed back into the domain) and then recovers as they reach an outflow face again.

In [ ]:
cfg_w = load_config(CONFIGS / "wind_change.json")
scn_w = scenario_from_dict(cfg_w["scenario"])
res_w = simulate(scn_w, cfg_w["q"], SolverOptions(**cfg_w["solver"]))
for w in scn_w.transport.wind:
    print(f"from t = {w.t_start:6.0f} s: wind ({w.u:+.1f}, {w.v:+.1f}) m/s")

fig, ax = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)
bw = res_w.budget
for name in ("emission", "removal", "inflow", "outflow"):
    ax[0].plot(bw["time"], bw[name], label=name)
for w in scn_w.transport.wind[1:]:
    ax[0].axvline(w.t_start, color="k", ls="--", lw=0.8)
ax[0].set(xlabel="t (s)", ylabel="microgram per output interval", title="Budget terms with changing wind")
ax[0].legend()
ax[1].plot(bw["time"], bw["inflow"], label="inflow")
for w in scn_w.transport.wind[1:]:
    ax[1].axvline(w.t_start, color="k", ls="--", lw=0.8)
ax[1].set(xlabel="t (s)", ylabel="microgram per output interval", title="Inflow term only (negative = loss through inflow faces)")
ax[1].legend()
show(fig)
show(plot_field_snapshots(res_w.concentration, times=[5400.0, 7800.0, 10200.0, 14400.0]))
dw = res_w.diagnostics
print(f"max |budget residual| {dw['max_abs_budget_residual']:.2e} microgram; negatives {dw['n_negative']}, "
      f"min concentration {dw['min_concentration']:.2e}")

## Optional interactive controls

Everything above runs without widgets. If `ipywidgets` is installed, the cell below adds sliders and an explicit
**Run** button (moving a slider does not recompute; the solve takes a couple of seconds). The plain function
`explore(...)` works anywhere.

In [ ]:
def explore(u=WIND_U, v=WIND_V, lam_factor=1.0, q_scale=1.0):
    '''Plain alternative to the widgets: run with a new wind, removal factor and overall source strength.'''
    scn = with_transport(wind=(WindSegment(0.0, u, v),), lam=lam_factor * LAMBDA)
    res = simulate(scn, [q_scale * q for q in Q], SOLVER)
    show(plot_field_snapshots(res.concentration, times=[3600.0, DURATION]))
    print(f"peak {res.concentration.max().item():.3g} microgram/m^3; max |budget residual| "
          f"{res.diagnostics['max_abs_budget_residual']:.2e} microgram")
    return res

try:
    import ipywidgets as W
    sliders = dict(u=W.FloatSlider(value=WIND_U, min=-2, max=2, step=0.1, description="u (m/s)"),
                   v=W.FloatSlider(value=WIND_V, min=-2, max=2, step=0.1, description="v (m/s)"),
                   lam_factor=W.FloatLogSlider(value=1.0, base=10, min=-1, max=1, step=0.1, description="lambda x"),
                   q_scale=W.FloatSlider(value=1.0, min=0.0, max=3.0, step=0.1, description="q scale"))
    button, out = W.Button(description="Run", button_style="primary"), W.Output()

    def _on_click(_):
        with out:
            out.clear_output(wait=True)
            explore(**{k: s.value for k, s in sliders.items()})

    button.on_click(_on_click)
    display(W.VBox([*sliders.values(), button, out]))
except ImportError:
    print("ipywidgets is not installed; call explore(u=..., v=..., lam_factor=..., q_scale=...) instead.")

In [ ]:
# plain, non-widget alternative (also the default path): change the numbers and re-run this cell
_ = explore(u=-0.5, v=0.5, lam_factor=1.0, q_scale=1.0)